# CUDA Tasks — Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Выполните ячейки сверху вниз
3. Git push и GitLab CI — **после** успешного прогона

Порядок: 01…08 (задача 06 после 05).


In [ ]:
!nvidia-smi
!nvcc --version

## Загрузка проекта

На локальной машине упакуйте папку `task-cuda` в zip и загрузите ниже.

In [ ]:
from google.colab import files
import zipfile, os, shutil

if os.path.isdir('task-cuda'):
    print('task-cuda уже есть')
else:
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name, 'r') as z:
        z.extractall('.')
    if not os.path.isdir('task-cuda'):
        for entry in os.listdir('.'):
            if os.path.isdir(entry) and os.path.isfile(os.path.join(entry, 'CMakeLists.txt')):
                shutil.move(entry, 'task-cuda')
                break
    assert os.path.isdir('task-cuda'), 'не найден каталог task-cuda после распаковки'
    print('OK:', os.listdir('task-cuda')[:8])

In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/smoke \
  runners/01-add.cu src/KernelAdd.cu src/CommonKernels.cu
/tmp/smoke --check

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_benchmark(csv_path, title):
    df = pd.read_csv(csv_path)
    if 'method' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for method, group in df.groupby('method'):
            size_df = group[group['mode'] == 'size']
            block_df = group[group['mode'] == 'block']
            if len(size_df):
                axes[0].plot(size_df['n'], size_df['ms'], marker='o', label=method)
            if len(block_df):
                axes[1].plot(block_df['block'], block_df['ms'], marker='o', label=method)
        axes[0].set_xlabel('n')
        axes[0].set_ylabel('ms')
        axes[0].set_title(f'{title}: time vs size')
        axes[0].grid(True)
        axes[0].legend()
        axes[1].set_xlabel('block')
        axes[1].set_ylabel('ms')
        axes[1].set_title(f'{title}: time vs block')
        axes[1].grid(True)
        axes[1].legend()
    else:
        size_df = df[df['mode'] == 'size']
        block_df = df[df['mode'] == 'block']
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        if len(size_df):
            axes[0].plot(size_df['n'], size_df['ms'], marker='o')
            axes[0].set_xlabel('n')
            axes[0].set_ylabel('ms')
            axes[0].set_title(f'{title}: time vs size')
            axes[0].grid(True)
        if len(block_df):
            axes[1].plot(block_df['block'], block_df['ms'], marker='o')
            axes[1].set_xlabel('block')
            axes[1].set_ylabel('ms')
            axes[1].set_title(f'{title}: time vs block')
            axes[1].grid(True)
    plt.tight_layout()
    plt.show()

## 01 — KernelAdd


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/01-add runners/01-add.cu src/KernelAdd.cu src/CommonKernels.cu

In [ ]:
! /tmp/01-add --check

In [ ]:
! /tmp/01-add --benchmark --out /tmp/01-add.csv

In [ ]:
plot_benchmark('/tmp/01-add.csv', '01 KernelAdd')

## 02 — KernelMul


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/02-mul runners/02-mul.cu src/KernelMul.cu src/CommonKernels.cu

In [ ]:
! /tmp/02-mul --check

In [ ]:
! /tmp/02-mul --benchmark --out /tmp/02-mul.csv

In [ ]:
plot_benchmark('/tmp/02-mul.csv', '02 KernelMul')

## 03 — MatrixAdd


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/03-matrix-add runners/03-matrix-add.cu src/KernelMatrixAdd.cu src/CommonKernels.cu

In [ ]:
! /tmp/03-matrix-add --check

In [ ]:
! /tmp/03-matrix-add --benchmark --out /tmp/03-matrix-add.csv

In [ ]:
plot_benchmark('/tmp/03-matrix-add.csv', '03 MatrixAdd')

## 04 — MatrixVectorMul


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/04-matrix-vector-mul runners/04-matrix-vector-mul.cu src/MatrixVectorMul.cu src/CommonKernels.cu

In [ ]:
! /tmp/04-matrix-vector-mul --check

In [ ]:
! /tmp/04-matrix-vector-mul --benchmark --out /tmp/04-matrix-vector-mul.csv

In [ ]:
plot_benchmark('/tmp/04-matrix-vector-mul.csv', '04 MatrixVectorMul')

## 05 — ScalarMul


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/05-scalar-mul runners/05-scalar-mul.cu src/ScalarMulRunner.cu src/ScalarMul.cu src/CommonKernels.cu

In [ ]:
! /tmp/05-scalar-mul --check

In [ ]:
! /tmp/05-scalar-mul --benchmark --out /tmp/05-scalar-mul.csv

In [ ]:
plot_benchmark('/tmp/05-scalar-mul.csv', '05 ScalarMul')

## 06 — CosineVector


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/06-cosine-vector runners/06-cosine-vector.cu src/CosineVector.cu src/ScalarMulRunner.cu src/ScalarMul.cu src/CommonKernels.cu

In [ ]:
! /tmp/06-cosine-vector --check

In [ ]:
! /tmp/06-cosine-vector --benchmark --out /tmp/06-cosine-vector.csv

In [ ]:
plot_benchmark('/tmp/06-cosine-vector.csv', '06 CosineVector')

## 07 — MatrixMul


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/07-matrix-mul runners/07-matrix-mul.cu src/MatrixMul.cu src/CommonKernels.cu

In [ ]:
! /tmp/07-matrix-mul --check

In [ ]:
! /tmp/07-matrix-mul --benchmark --out /tmp/07-matrix-mul.csv

In [ ]:
plot_benchmark('/tmp/07-matrix-mul.csv', '07 MatrixMul')

## 08 — Filter


In [ ]:
%%bash
cd task-cuda
nvcc -O3 -arch=sm_75 -I include -o /tmp/08-filter runners/08-filter.cu src/Filter.cu

In [ ]:
! /tmp/08-filter --check

In [ ]:
! /tmp/08-filter --benchmark --out /tmp/08-filter.csv

In [ ]:
plot_benchmark('/tmp/08-filter.csv', '08 Filter')